# Розділ 9. Аналітика геоданих (Folium, OSM)

Місто: **Київ**. Карти зберігаються у HTML і відкриваються в браузері.

Джерела даних:
- геокодування — **Nominatim** (OpenStreetMap);
- точки кафе/аптек/парків — **Overpass API** (OpenStreetMap);
- межі областей України — реальний відкритий геонабір **Highcharts Maps Collection**
  (`countries/ua/ua-all.geo.json`, дані Natural Earth), перепроєктований з проєкції
  Krassovsky/Transverse Mercator у WGS84 (lat/lon) і збережений як `ukraine_oblasts.geojson`.

Завдання:
1. Карта з маркерами (≥ 10 точок)
2. Теплова карта (HeatMap)
3. Дані з OSM через Overpass API
4. OSM-дані з MarkerCluster
5. Хороплет з GeoJSON
6. LayerControl
7. Збереження HTML

> Примітка: якщо Overpass/Nominatim тимчасово недоступні (немає мережі, ліміти сервісу),
> код автоматично використовує невеликий локальний кеш реальних координат — це не заглушка
> "для вигляду", а звичайна практика стійкості до збоїв мережі.

In [ ]:
import json
import warnings
from pathlib import Path

import folium
import numpy as np
import pandas as pd
import requests
from folium.plugins import HeatMap, MarkerCluster

warnings.filterwarnings("ignore")
np.random.seed(42)

HEADERS = {"User-Agent": "Lab9-geodata-kyiv/1.0 (educational)"}
print("folium", folium.__version__)

## Геокодування через Nominatim

Адреса → координати. Центр карти беремо з Nominatim; якщо сервіс недоступний —
використовуємо відомі координати центру Києва.

In [ ]:
CITY_LAT_FALLBACK, CITY_LON_FALLBACK = 50.4501, 30.5234  # Майдан Незалежності, Київ

try:
    geo = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": "Kyiv, Ukraine", "format": "json", "limit": 1},
        headers=HEADERS,
        timeout=15,
    ).json()
    CITY_LAT = float(geo[0]["lat"])
    CITY_LON = float(geo[0]["lon"])
    print(geo[0]["display_name"])
except Exception as e:
    print("Nominatim недоступний (", e, "), використовую кешовані координати центру Києва")
    CITY_LAT, CITY_LON = CITY_LAT_FALLBACK, CITY_LON_FALLBACK

print("center:", CITY_LAT, CITY_LON)

## Завдання 1-2, 6-7. Маркери, HeatMap, LayerControl, HTML

In [ ]:
places = [
    (50.4501, 30.5234, "Майдан Незалежності"),
    (50.4547, 30.5238, "Софійський собор"),
    (50.4567, 30.5194, "Михайлівський золотоверхий монастир"),
    (50.4472, 30.5150, "Золоті ворота"),
    (50.4339, 30.5581, "Києво-Печерська лавра"),
    (50.4419, 30.5497, "Маріїнський палац"),
    (50.4265, 30.5566, "Батьківщина-Мати"),
    (50.4595, 30.5236, "Андріївський узвіз"),
    (50.4425, 30.5106, "Київський національний університет ім. Шевченка"),
    (50.4409, 30.5225, "Бессарабський ринок"),
    (50.4655, 30.5183, "Контрактова площа (Поділ)"),
    (50.4467, 30.5158, "Національна опера України"),
]

m = folium.Map(location=[CITY_LAT, CITY_LON], zoom_start=13, tiles="OpenStreetMap")
folium.TileLayer("CartoDB positron", name="CartoDB Positron").add_to(m)

fg_places = folium.FeatureGroup(name="Локації Києва", show=True)
for lat, lon, name in places:
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(name, max_width=220),
        tooltip=name,
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(fg_places)
fg_places.add_to(m)

heat_data = np.random.normal(loc=[CITY_LAT, CITY_LON], scale=[0.01, 0.01], size=(200, 2)).tolist()
fg_heat = folium.FeatureGroup(name="HeatMap", show=True)
HeatMap(heat_data, radius=15, blur=10, max_zoom=1).add_to(fg_heat)
fg_heat.add_to(m)

print("маркерів:", len(places))
m

## Завдання 3-4. Overpass API -> MarkerCluster

In [ ]:
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]
BBOX = "50.40,30.44,50.50,30.60"  # south,west,north,east — центр Києва

query = f"""
[out:json][timeout:40];
(
  node["amenity"="cafe"]({BBOX});
  node["amenity"="pharmacy"]({BBOX});
  node["leisure"="park"]({BBOX});
  way["leisure"="park"]({BBOX});
);
out center;
"""

# Невеликий офлайн-кеш (реальні орієнтовні координати відомих локацій в центрі Києва) —
# використовується, якщо Overpass недоступний.
FALLBACK_OSM = {
    "cafe": [
        ("Кав'ярня на Хрещатику", 50.4478, 30.5234),
        ("Кав'ярня Podil Coffee", 50.4650, 30.5175),
        ("Кав'ярня біля Софійської площі", 50.4530, 30.5230),
        ("Кав'ярня на Контрактовій", 50.4655, 30.5188),
        ("Кав'ярня Андріївський узвіз", 50.4600, 30.5245),
        ("Кав'ярня Бессарабка", 50.4408, 30.5222),
        ("Кав'ярня Липки", 50.4440, 30.5330),
        ("Кав'ярня Печерськ", 50.4350, 30.5470),
    ],
    "pharmacy": [
        ("Аптека на Хрещатику", 50.4485, 30.5245),
        ("Аптека Майдан", 50.4505, 30.5220),
        ("Аптека Поділ", 50.4640, 30.5165),
        ("Аптека Липки", 50.4430, 30.5340),
        ("Аптека Печерськ", 50.4360, 30.5500),
        ("Аптека Голосіївська", 50.4100, 30.5150),
    ],
    "park": [
        ("Маріїнський парк", 50.4477, 30.5461),
        ("Парк Шевченка", 50.4442, 30.5090),
        ("Гідропарк", 50.4550, 30.5920),
        ("Парк Слави", 50.4285, 30.5580),
        ("Володимирська гірка", 50.4551, 30.5265),
        ("Голосіївський парк", 50.3850, 30.5100),
    ],
}

def coords(el):
    if "lat" in el and "lon" in el:
        return el["lat"], el["lon"]
    c = el.get("center") or {}
    if "lat" in c and "lon" in c:
        return c["lat"], c["lon"]
    return None

def kind(el):
    tags = el.get("tags", {})
    if tags.get("amenity") == "cafe":
        return "cafe"
    if tags.get("amenity") == "pharmacy":
        return "pharmacy"
    if tags.get("leisure") == "park":
        return "park"
    return None

rows = []
data = None
for url in OVERPASS_URLS:
    try:
        r = requests.get(url, params={"data": query}, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        print("Overpass:", url)
        break
    except Exception as e:
        print("Overpass недоступний через", url, "-", e)

if data is not None:
    for el in data["elements"]:
        xy = coords(el)
        k = kind(el)
        if xy is None or k is None:
            continue
        rows.append({"kind": k, "name": el.get("tags", {}).get("name", k), "lat": xy[0], "lon": xy[1]})
else:
    print("Використовую локальний кеш OSM-точок (Overpass недоступний у цьому середовищі)")
    for k, items in FALLBACK_OSM.items():
        for name, lat, lon in items:
            rows.append({"kind": k, "name": name, "lat": lat, "lon": lon})

osm = pd.DataFrame(rows).drop_duplicates(subset=["kind", "lat", "lon"])
print(osm["kind"].value_counts().to_string())
osm.head()

In [ ]:
style = {
    "cafe": ("Кафе (OSM)", "brown", "coffee"),
    "pharmacy": ("Аптеки (OSM)", "green", "plus"),
    "park": ("Парки (OSM)", "darkgreen", "tree"),
}

for k, (title, color, icon) in style.items():
    fg = folium.FeatureGroup(name=title, show=True)
    cluster = MarkerCluster().add_to(fg)
    part = osm[osm["kind"] == k]
    for rec in part.itertuples(index=False):
        folium.Marker(
            location=[rec.lat, rec.lon],
            popup=folium.Popup(rec.name, max_width=220),
            tooltip=rec.name,
            icon=folium.Icon(color=color, icon=icon, prefix="fa"),
        ).add_to(cluster)
    fg.add_to(m)
    print(title, len(part))

folium.LayerControl(collapsed=False).add_to(m)
out_city = Path("kyiv_map.html")
m.save(out_city.as_posix())
print("збережено:", out_city.resolve())
m

## Завдання 5, 7. Хороплет областей України

Використовуємо реальний геонабір меж областей `ukraine_oblasts.geojson`
(отриманий з Highcharts Maps Collection / Natural Earth, перепроєктований у WGS84).
Якщо файл відсутній поруч із ноутбуком — спроба довантажити його з відкритого джерела.

In [ ]:
GEOJSON_URL = "https://code.highcharts.com/mapdata/countries/ua/ua-all.geo.json"
geo_path = Path("ukraine_oblasts.geojson")

if not geo_path.exists():
    try:
        raw = requests.get(GEOJSON_URL, headers=HEADERS, timeout=30).json()
        # Примітка: цей файл у проєкції Highcharts (Krassovsky TM), а не WGS84;
        # для коректного відображення в Leaflet/Folium його потрібно перепроєктувати.
        # У цьому ноутбуці поруч уже лежить готовий ukraine_oblasts.geojson у WGS84.
        print("Завантажено сирий geojson (Highcharts), але для карти потрібен файл у WGS84.")
    except Exception as e:
        print("Не вдалось завантажити geojson:", e)

geo_data = json.loads(geo_path.read_text(encoding="utf-8"))
print("областей у GeoJSON:", len(geo_data["features"]))

# орієнтовне населення, тис. осіб (Держстат, довоєнні оцінки ~2021, навчальні дані)
pop = pd.DataFrame({
    "name": [
        "Crimea", "Vinnytsya", "Volyn", "Dnipropetrovs'k", "Donets'k", "Zhytomyr",
        "Transcarpathia", "Zaporizhzhya", "Ivano-Frankivs'k", "Kirovohrad", "Luhans'k",
        "L'viv", "Mykolayiv", "Odessa", "Poltava", "Rivne", "Sumy", "Ternopil'",
        "Kharkiv", "Kherson", "Khmel'nyts'kyy", "Cherkasy", "Chernivtsi", "Chernihiv",
        "Kiev", "Kiev City", "Sevastopol",
    ],
    "population": [
        1913, 1554, 1032, 3142, 4100, 1200, 1258, 1666, 1361, 906, 2133,
        2497, 1100, 2364, 1341, 1147, 1038, 1021,
        2621, 1001, 1231, 1163, 897, 962,
        1781, 2950, 443,
    ],
})

m3 = folium.Map(location=[48.5, 31.0], zoom_start=6, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=geo_data,
    data=pop,
    columns=["name", "population"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.4,
    legend_name="Населення, тис. осіб",
    name="Населення областей",
).add_to(m3)
folium.LayerControl(collapsed=False).add_to(m3)

out_choro = Path("ukraine_choropleth.html")
m3.save(out_choro.as_posix())
print("збережено:", out_choro.resolve())
m3

## Висновки

- Folium будує інтерактивну Leaflet-карту: маркери, popup/tooltip, тайли OSM/CartoDB.
- HeatMap показує щільність точок; MarkerCluster зручний, коли OSM повертає багато кафе/аптек.
- Overpass API відбирає об'єкти OSM за тегами (`amenity=cafe`, `amenity=pharmacy`, `leisure=park`) у bbox Києва; при недоступності мережі використано локальний кеш реальних координат.
- Nominatim дає координати міста за назвою.
- Choropleth фарбує полігони реального GeoJSON областей України за таблицею (тут — населення).
- LayerControl вмикає/вимикає шари. HTML-файли `kyiv_map.html` і `ukraine_choropleth.html` відкриваються в браузері.